In [0]:

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 50 * 1024 * 1024)
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.join.preferSortMergeJoin", "true")
spark.conf.set("spark.sql.shuffle.partitions", "100")

In [0]:

# Import required libraries
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import json
import re

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", StringType(), True),
    StructField("order_approved_at", StringType(), True),
    StructField("order_delivered_carrier_date", StringType(), True),
    StructField("order_delivered_customer_date", StringType(), True),
    StructField("order_estimated_delivery_date", StringType(), True)
])

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True)
])

customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("date_of_birth", StringType(), True)
        
])

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

try:
    
    df_orders = (
        spark.read
        .schema(orders_schema)
        .option("header", True)
        .option("dateFormat", "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .csv("gs://gds-external-bucket-for-databrick/ecom_data/orders.csv")
    )

    # Add processing metadata
    df_orders = (
        df_orders
        .withColumn("processed_timestamp", F.current_timestamp())
        .withColumn("batch_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
        .withColumn("source_system", F.lit("ecommerce_orders"))
    )

    df_orders.cache()

    metrics = df_orders.select(
        F.count("order_id").alias("total_records"),
        F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_ids"),
        F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids")
    ).collect()[0]

    print(metrics)

    valid_condition = (
        (F.col("order_id").isNotNull()) &
        (F.col("customer_id").isNotNull())
    )

    df_valid_orders = df_orders.filter(valid_condition)
    df_invalid_orders = df_orders.filter(~valid_condition)

    print("Valid records:", df_valid_orders.count())
    print("Invalid records:", df_invalid_orders.count())

except Exception as e:
    print(f"Error reading orders data: {str(e)}")
    raise

Row(total_records=397260, null_order_ids=0, null_customer_ids=198630)
Valid records: 198630
Invalid records: 198630


In [0]:
from pyspark.sql import functions as F
from datetime import datetime

try:

    df_order_items = (
        spark.read
        .schema(order_items_schema)
        .option("header", True)
        .option("dateFormat", "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .csv("gs://gds-external-bucket-for-databrick/ecom_data/order_items.csv")
    )

 # Add processing metadata
    df_order_items = (
        df_order_items
        .withColumn("processed_timestamp", F.current_timestamp())
        .withColumn("batch_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
        .withColumn("source_system", F.lit("ecommerce_orders"))
    )

    df_order_items.cache()

    metrics = df_order_items.select(
        F.count("order_id").alias("total_records"),
        F.sum(F.col("order_id").isNull().cast("int")).alias("null_order_ids"),
        F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
        F.sum((F.col("price") <= 0).cast("int")).alias("invalid_amounts")
    ).collect()[0]

    print(metrics)

    valid_condition = (
        (F.col("order_id").isNotNull()) &
        (F.col("customer_id").isNotNull()) &
        (F.col("price") > 0)
    )

    df_valid_order_items = df_order_items.filter(valid_condition)
    df_invalid_order_items = df_order_items.filter(~valid_condition)

    print("Valid records:", df_valid_order_items.count())
    print("Invalid records:", df_invalid_order_items.count())

except Exception as e:
    print(f"Error reading orders data: {str(e)}")
    raise

Row(total_records=453466, null_order_ids=0, null_customer_ids=0, invalid_amounts=1014)
Valid records: 452452
Invalid records: 1014


In [0]:
from pyspark.sql import functions as F
from datetime import datetime

try:
    df_customers = (
        spark.read
        .schema(customers_schema) 
        .option("header", True)
        .csv("gs://gds-external-bucket-for-databrick/ecom_data/customers.csv")
    )

    df_customers = df_customers.withColumn(
        "date_of_birth",
        F.try_to_date(
            F.when(
                (F.col("date_of_birth") == "NA") | (F.col("date_of_birth").isNull()),
                None
            ).otherwise(F.col("date_of_birth")),
            "dd-MMM-yy"
        )
    )

    df_customers = (
        df_customers
        .withColumn("processed_timestamp", F.current_timestamp())
        .withColumn("batch_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
        .withColumn("source_system", F.lit("ecommerce_customers"))
    )

    df_customers.cache()

    df_customers.printSchema()

    metrics = df_customers.select(
        F.count("*").alias("total_records"),
        F.sum(F.col("customer_id").isNull().cast("int")).alias("null_customer_ids"),
        F.sum(F.col("email").isNull().cast("int")).alias("null_emails"),
        F.sum(F.col("phone").isNull().cast("int")).alias("null_phones"),
        F.sum((F.col("date_of_birth") > F.current_date()).cast("int")).alias("future_birth_dates")
    ).collect()[0]

    print(metrics)

except Exception as e:
    print(f"Error reading customers data: {str(e)}")
    raise

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- processed_timestamp: timestamp (nullable = false)
 |-- batch_id: string (nullable = false)
 |-- source_system: string (nullable = false)

Row(total_records=198853, null_customer_ids=0, null_emails=0, null_phones=0, future_birth_dates=198772)


In [0]:
from pyspark.sql.functions import *

orders_clean = df_orders.withColumn(
    "order_ts",
     to_timestamp(col("order_purchase_timestamp"), "dd-MM-yyyy HH:mm")
)

In [0]:
# part 3
from pyspark.sql.functions import col, sum, count

order_items_agg = (
    df_order_items
    .select("order_id", "price")
    .filter(
        (col("order_id").isNotNull()) &
        (col("price") > 0)
    ).groupBy("order_id")
    .agg(
        sum("price").alias("total_amount"),
        count("order_id").alias("total_items")  
    )
)

order_items_agg.show()

+--------------------+------------+-----------+
|            order_id|total_amount|total_items|
+--------------------+------------+-----------+
|0016dfedd97fc2950...|       398.0|          8|
|0071ee2429bc1efdc...|      719.92|          4|
|014405982914c2cde...|      196.92|          8|
|019886de8f385a39b...|       639.6|          4|
|01a26a995dcef1d58...|     1199.96|          4|
|01a6ad782455876aa...|      139.96|          4|
|01d907b3e209269e1...|      607.96|          4|
|01f9f30f42a15cb09...|       199.8|          4|
|024ab991361d48e49...|      319.92|          4|
|024b3b6b53b99a1eb...|      1399.6|          4|
|0281523bfda80130f...|       143.6|          4|
|028dc52e12ddda803...|      199.96|          4|
|02a065131a2d2b72b...|      2500.0|         20|
|02cbfc1aa8d209d7f...|      111.96|          4|
|034e23cd76658cb8d...|       279.6|          4|
|036dd381dfb3ec75e...|       220.0|          4|
|03ebfa9712b7dbc70...|       187.6|          4|
|0420da8d50a378401...|       239.6|     

In [0]:
df_valid_orders.show(5)

+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+----------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date| processed_timestamp|       batch_id|   source_system|
+--------------------+--------------------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+--------------------+---------------+----------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|        02-10-2017 10:56| 02-10-2017 11:07|            04-10-2017 19:55|             10-10-2017 21:25|             18-10-2017 00:00|2026-02-24 21:08:...|20260224_210823|ecommerce_orders|
|53cdb2fc8bc7dce0b...|b0830f

In [0]:
from pyspark.sql.functions import col, to_date, broadcast

orders_fact = (
    df_valid_orders
    .join(broadcast(order_items_agg), "order_id", "inner")
    .withColumn(
        "order_date",
        to_date(col("order_purchase_timestamp"), "dd-MM-yyyy HH:mm")
    )
    .select(
        "order_id",
        "customer_id",
        "order_date",
        "total_amount",
        "total_items",
        "order_status"
    )
)

orders_fact.show(5, False)

+--------------------------------+--------------------------------+----------+------------+-----------+------------+
|order_id                        |customer_id                     |order_date|total_amount|total_items|order_status|
+--------------------------------+--------------------------------+----------+------------+-----------+------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|2017-10-02|119.96      |4          |delivered   |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|2018-07-24|474.8       |4          |delivered   |
|47770eb9100c2d0c44946d9cf07ec65d|41ce2a54c0b03bf3443c3d931a367089|2018-08-08|639.6       |4          |delivered   |
|949d5b44dbf5de918fe9c16f97b45f8a|f88197465ea7920adcdbec7375364d82|2017-11-18|180.0       |4          |delivered   |
|ad21c59c0840e6cb83a9ceb5573f8159|8ab97904e6daea8866dbdbc4fb7aad2c|2018-02-13|79.6        |4          |delivered   |
+--------------------------------+------------------------------

In [0]:
from pyspark.sql.functions import avg, min, max, count, sum, col

customer_metrics = (
    orders_fact
    .select(
        "customer_id",
        "order_id",
        "total_amount",
        "order_date"
    )
    .filter(
        (col("total_amount") > 0) 
      #  (col("order_date") >= date_sub(current_date(), 365))
    )
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("total_amount").alias("lifetime_value"),
        avg("total_amount").alias("avg_order_value"),
        min("order_date").alias("first_order_date"),
        max("order_date").alias("last_order_date")
    )
)

customer_metrics.show(5)

+--------------------+------------+--------------+---------------+----------------+---------------+
|         customer_id|total_orders|lifetime_value|avg_order_value|first_order_date|last_order_date|
+--------------------+------------+--------------+---------------+----------------+---------------+
|f54a9f0e6b351c431...|           2|         159.2|           79.6|      2017-01-23|     2017-01-23|
|2a1dfb647f32f4390...|           2|        3560.0|         1780.0|      2018-06-01|     2018-06-01|
|4f28355e5c17a4a42...|           2|        1112.0|          556.0|      2017-05-18|     2017-05-18|
|4632eb5a8f175f6fe...|           2|         639.2|          319.6|      2017-11-30|     2017-11-30|
|843ff05b30ce4f75b...|           2|         679.2|          339.6|      2017-11-13|     2017-11-13|
+--------------------+------------+--------------+---------------+----------------+---------------+
only showing top 5 rows


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

schema = StructType() \
    .add("order_id", StringType()) \
    .add("customer_id", StringType()) \
    .add("order_ts", StringType()) \
    .add("total_amount", DoubleType()) \
    .add("order_status", StringType())

bronze_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("pathGlobFilter", "orders_day_*.csv")
    .schema(schema)
    .load("gs://gds-external-bucket-for-databrick/ecom_data/")
)

bronze_query = (
    bronze_df
    .coalesce(2)
    .writeStream
    .format("delta")
    .option("checkpointLocation", "gs://gds-external-bucket-for-databrick/checkpoints/bronze")
    .outputMode("append")
    .trigger(availableNow=True)
    .start("gs://gds-external-bucket-for-databrick/bronze/orders")
)

bronze_query.awaitTermination()

In [0]:
display(dbutils.fs.ls(
    "gs://gds-external-bucket-for-databrick/bronze/"
))

path,name,size,modificationTime
gs://gds-external-bucket-for-databrick/bronze/orders/,orders/,0,0


In [0]:
from pyspark.sql.functions import broadcast, col, expr, to_date

silver_df = (
    spark.readStream
    .format("delta")
    .load("gs://gds-external-bucket-for-databrick/bronze/orders")
)

orders_fact = (
    silver_df
    .select("order_id", "customer_id", "order_ts", "total_amount", "order_status")
    .join(
        broadcast(order_items_agg.select("order_id", "total_items")),
        "order_id",
        "inner"
    )
    .withColumn(
        "order_date",
        to_date(expr("try_to_timestamp(order_ts)"))
    )
)

silver_query = (
    orders_fact
    .coalesce(2) 
    .writeStream
    .format("delta")
    .option("checkpointLocation", "gs://gds-external-bucket-for-databrick/checkpoints/silver")
    .outputMode("append")
    .trigger(availableNow=True) 
    .start("gs://gds-external-bucket-for-databrick/silver/orders_fact")
)

silver_query.awaitTermination()

In [0]:
# part 5

orders_fact.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(order_date, 'to_date('try_to_timestamp('order_ts)), None)]
+- ~Project [order_id#14730, customer_id#14731, order_ts#14732, total_amount#14733, order_status#14734, total_items#3996L]
   +- ~Join Inner, (order_id#14730 = order_id#1137)
      :- ~Project [order_id#14730, customer_id#14731, order_ts#14732, total_amount#14733, order_status#14734]
      :  +- ~StreamingRelation DataSource(org.apache.spark.sql.classic.SparkSession@220bfea6,delta,List(),None,List(),None,Map(path -> gs://gds-external-bucket-for-databrick/bronze/orders),None), tahoe, [order_id#14730, customer_id#14731, order_ts#14732, total_amount#14733, order_status#14734]
      +- ResolvedHint (strategy=broadcast)
         +- Project [order_id#1137, total_items#3996L]
            +- Aggregate [order_id#1137], [order_id#1137, sum(price#1143) AS total_amount#3995, count(order_id#1137) AS total_items#3996L]
               +- Filter (isnotnull(order_id#1137) AND (price#

In [0]:
# ---------------- Performance Optimization ----------------

# Shuffle Identification:
# Shuffle is indicated by Exchange hashpartitioning() in the physical plan.
# Shuffle occurs in the following stages:
# 1. groupBy(order_id) — required to aggregate total_amount and total_items.
# 2. repartition(order_date) — required to partition output data for storage.
#
# These are wide transformations because Spark redistributes data across partitions.
#
# Broadcast Join Optimization:
# Execution plan shows "BroadcastHashJoin", meaning the smaller dataset
# (order_items_agg) was broadcast to all executors. This avoids shuffle on the
# broadcast side and reduces network I/O, improving join performance.
#
# Adaptive Query Execution:
# Adaptive execution is enabled, allowing Spark to dynamically optimize shuffle
# partitions and execution strategy at runtime.
#
# Overall Optimization Strategy:
# - Aggregated data before join to reduce dataset size
# - Used broadcast join to minimize shuffle during join
# - Partitioned output by order_date for efficient storage and query pruning
# - Enabled adaptive execution for runtime optimization
#
# ------------------------------------------------------------------

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, count, sum, avg, min, max

gold_path = "gs://gds-external-bucket-for-databrick/gold/customer_metrics"

def upsert_to_gold(microBatchDF, batchId):

    base_df = (
        microBatchDF
        .filter(col("total_amount") > 0)
        .select("customer_id", "order_id", "total_amount", "order_date")
    )
    agg_df = (
        base_df
        .groupBy("customer_id")
        .agg(
            count("order_id").alias("total_orders"),
            sum("total_amount").alias("lifetime_value"),
            avg("total_amount").alias("avg_order_value"),
            min("order_date").alias("first_order_date"),
            max("order_date").alias("last_order_date")
        )
    )

    if DeltaTable.isDeltaTable(spark, gold_path):
        gold_table = DeltaTable.forPath(spark, gold_path)

        (gold_table.alias("t")
         .merge(
            agg_df.alias("s"),
            "t.customer_id = s.customer_id"
         )
         .whenMatchedUpdate(set={
            "total_orders": "t.total_orders + s.total_orders",
            "lifetime_value": "t.lifetime_value + s.lifetime_value",
            "avg_order_value": "s.avg_order_value",
            "first_order_date": "least(t.first_order_date, s.first_order_date)",
            "last_order_date": "greatest(t.last_order_date, s.last_order_date)"
         })
         .whenNotMatchedInsertAll()
         .execute()
        )

    else:
        (agg_df
         .coalesce(1)
         .write
         .format("delta")
         .mode("overwrite")
         .save(gold_path)
        )

In [0]:
gold_stream = (
    spark.readStream
    .format("delta")
    .load("gs://gds-external-bucket-for-databrick/silver/orders_fact")

    .repartition(4)

    .writeStream
    .foreachBatch(lambda df, batchId: (
        upsert_to_gold(df, batchId) if not df.isEmpty() else None
    ))

    .outputMode("update") 

    .trigger(availableNow=True) 

    .option(
        "checkpointLocation",
        "gs://gds-external-bucket-for-databrick/checkpoints/gold"
    )

    .start()
)

gold_stream.awaitTermination()